In [13]:
import pandas as pd
import numpy as np

### Load Data

In [14]:
# Load the data
btc = pd.read_csv("../datasets/BTC_full_data.csv")
eth = pd.read_csv("../datasets/ETH_full_data.csv")

print(btc.head())
print(eth.head())

         Date          Open          High           Low         Close  \
0  2018-01-01  14112.200195  14112.200195  13154.700195  13657.200195   
1  2018-01-02  13625.000000  15444.599609  13163.599609  14982.099609   
2  2018-01-03  14978.200195  15572.799805  14844.500000  15201.000000   
3  2018-01-04  15270.700195  15739.700195  14522.200195  15599.200195   
4  2018-01-05  15477.200195  17705.199219  15202.799805  17429.500000   

      Adj Close       Volume  Log_Return  
0  13657.200195  10291200000         NaN  
1  14982.099609  16846600192    0.092589  
2  15201.000000  16871900160    0.014505  
3  15599.200195  21783199744    0.025858  
4  17429.500000  23840899072    0.110945  
         Date        Open         High         Low       Close   Adj Close  \
0  2018-01-01  755.757019   782.530029  742.004028  772.640991  772.640991   
1  2018-01-02  772.346008   914.830017  772.346008  884.443970  884.443970   
2  2018-01-03  886.000000   974.471008  868.450989  962.719971  962.7

In [15]:
def cusum_mean_reversion(df, window=20, h=0.05):
    df = df.copy()

    # rolling mean of returns
    df["mean_ret"] = df["Log_Return"].rolling(window).mean()

    # demeaned returns
    df["demeaned"] = df["Log_Return"] - df["mean_ret"]

    # rolling cumulative sum
    df["CUSUM"] = df["demeaned"].rolling(window).sum()

    position = []
    current_pos = 0

    for i in range(len(df)):

        c = df["CUSUM"].iloc[i]

        if np.isnan(c):
            position.append(0)
            continue

        # entry
        if current_pos == 0:
            if c > h:
                current_pos = -1
            elif c < -h:
                current_pos = 1

        # exit when back near zero
        elif current_pos == 1:
            if c >= 0:
                current_pos = 0

        elif current_pos == -1:
            if c <= 0:
                current_pos = 0

        position.append(current_pos)

    df["position"] = position

    df["strategy_return"] = df["position"].shift(1) * df["Log_Return"]

    return df

In [16]:
btc_cusum = cusum_mean_reversion(btc, window=20, h=0.05)
eth_cusum = cusum_mean_reversion(eth, window=20, h=0.05)

### Evaluation
We will evaluate the strategy using 6 metrics:
1. Cumulative PnL
2. Average Daily PnL
3. Volatility
4. Annualised Return
5. Sharpe Ratio
6. Max Drawdown

In [17]:
def evaluate_strategy(strategy_returns):
    r = strategy_returns.dropna()

    # cumulative pnl
    cumulative_pnl = np.exp(r.cumsum()).iloc[-1] - 1
    # average daily pnl
    avg_daily_pnl = r.mean()
    # volatility
    volatility = r.std()
    # annualised return
    annual_return = r.mean() * 365
    # annualised volatility
    annual_vol = volatility * np.sqrt(365)
    # sharpe ratio
    rf = 0.03
    sharpe = (annual_return - rf) / annual_vol
    # max drawdown
    cumulative = np.exp(r.cumsum())
    peak = cumulative.cummax()
    drawdown = (cumulative - peak) / peak
    max_dd = drawdown.min()

    return {
        "Cumulative PnL": cumulative_pnl,
        "Average Daily PnL": avg_daily_pnl,
        "Volatility": volatility,
        "Annualised Return": annual_return,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": max_dd
    }


In [18]:
btc_results = evaluate_strategy(btc_cusum["strategy_return"])
eth_results = evaluate_strategy(eth_cusum["strategy_return"])

print("BTC Results")
print(btc_results)

print("\nETH Results")
print(eth_results)

BTC Results
{'Cumulative PnL': np.float64(-0.9089535443751757), 'Average Daily PnL': np.float64(-0.0008206799319709213), 'Volatility': np.float64(0.030110753221868174), 'Annualised Return': np.float64(-0.29954817516938625), 'Sharpe Ratio': np.float64(-0.5728631139135116), 'Max Drawdown': np.float64(-0.9591145631730682)}

ETH Results
{'Cumulative PnL': np.float64(-0.8833729863305655), 'Average Daily PnL': np.float64(-0.0007358816280046157), 'Volatility': np.float64(0.04024836080052995), 'Annualised Return': np.float64(-0.26859679422168475), 'Sharpe Ratio': np.float64(-0.3883206724342403), 'Max Drawdown': np.float64(-0.9885082756635282)}


### Results Interpretation
BTC:
1. Cumulative PnL: -0.90 -> The portfolio lost 90 times its initial value
2. Average Daily PnL: -0.082% avarage return per day
3. Volatility: 3.01%
4. Annualised Return: -0.3 -> 30% per year
5. Sharpe Ratio: -0.57
6. Max Drawdown: -0.96

ETH:
1. Cumulative PnL: -0.88 -> The portfolio lost 88 times its initial value
2. Average Daily PnL: -0.074% avarage return per day
3. Volatility: 4.02%
4. Annualised Return: -0.27 -> -27% per year
5. Sharpe Ratio: -0.39
6. Max Drawdown: -0.99

### Trying other window size

### Window=30

In [19]:
def cusum_mean_reversion30(df, window=30, h=0.05):
    df = df.copy()

    # rolling mean of returns
    df["mean_ret"] = df["Log_Return"].rolling(window).mean()

    # demeaned returns
    df["demeaned"] = df["Log_Return"] - df["mean_ret"]

    # rolling cumulative sum
    df["CUSUM"] = df["demeaned"].rolling(window).sum()

    position = []
    current_pos = 0

    for i in range(len(df)):

        c = df["CUSUM"].iloc[i]

        if np.isnan(c):
            position.append(0)
            continue

        # entry
        if current_pos == 0:
            if c > h:
                current_pos = -1
            elif c < -h:
                current_pos = 1

        # exit when back near zero
        elif current_pos == 1:
            if c >= 0:
                current_pos = 0

        elif current_pos == -1:
            if c <= 0:
                current_pos = 0

        position.append(current_pos)

    df["position"] = position

    df["strategy_return"] = df["position"].shift(1) * df["Log_Return"]

    return df

In [20]:
btc_cusum30 = cusum_mean_reversion30(btc, window=30, h=0.05)
eth_cusum30 = cusum_mean_reversion30(eth, window=30, h=0.05)

In [21]:
btc_results30 = evaluate_strategy(btc_cusum30["strategy_return"])
eth_results30 = evaluate_strategy(eth_cusum30["strategy_return"])

print("BTC 30 Results")
print(btc_results30)

print("\nETH 30 Results")
print(eth_results30)

BTC 30 Results
{'Cumulative PnL': np.float64(-0.8903575469242176), 'Average Daily PnL': np.float64(-0.0007570310390017806), 'Volatility': np.float64(0.03084636211872341), 'Annualised Return': np.float64(-0.2763163292356499), 'Sharpe Ratio': np.float64(-0.5197802385411233), 'Max Drawdown': np.float64(-0.9503900777888208)}

ETH 30 Results
{'Cumulative PnL': np.float64(-0.9976174412349615), 'Average Daily PnL': np.float64(-0.002068349403238489), 'Volatility': np.float64(0.041552326066940176), 'Annualised Return': np.float64(-0.7549475321820486), 'Sharpe Ratio': np.float64(-0.9887781388227126), 'Max Drawdown': np.float64(-0.9987145246021235)}


### Window =60

In [22]:
def cusum_mean_reversion60(df, window=60, h=0.05):
    df = df.copy()

    # rolling mean of returns
    df["mean_ret"] = df["Log_Return"].rolling(window).mean()

    # demeaned returns
    df["demeaned"] = df["Log_Return"] - df["mean_ret"]

    # rolling cumulative sum
    df["CUSUM"] = df["demeaned"].rolling(window).sum()

    position = []
    current_pos = 0

    for i in range(len(df)):

        c = df["CUSUM"].iloc[i]

        if np.isnan(c):
            position.append(0)
            continue

        # entry
        if current_pos == 0:
            if c > h:
                current_pos = -1
            elif c < -h:
                current_pos = 1

        # exit when back near zero
        elif current_pos == 1:
            if c >= 0:
                current_pos = 0

        elif current_pos == -1:
            if c <= 0:
                current_pos = 0

        position.append(current_pos)

    df["position"] = position

    df["strategy_return"] = df["position"].shift(1) * df["Log_Return"]

    return df

In [25]:
btc_cusum60 = cusum_mean_reversion60(btc, window=60, h=0.05)
eth_cusum60 = cusum_mean_reversion60(eth, window=60, h=0.05)

In [26]:
btc_results60 = evaluate_strategy(btc_cusum60["strategy_return"])
eth_results60 = evaluate_strategy(eth_cusum60["strategy_return"])

print("BTC 60 Results")
print(btc_results60)

print("\nETH 60 Results")
print(eth_results60)

BTC 60 Results
{'Cumulative PnL': np.float64(-0.907934011362016), 'Average Daily PnL': np.float64(-0.0008168663326173091), 'Volatility': np.float64(0.030926365541881922), 'Annualised Return': np.float64(-0.29815621140531784), 'Sharpe Ratio': np.float64(-0.5553992795031248), 'Max Drawdown': np.float64(-0.9617305194586352)}

ETH 60 Results
{'Cumulative PnL': np.float64(-0.7762845827432885), 'Average Daily PnL': np.float64(-0.0005128015388764726), 'Volatility': np.float64(0.04002089450334745), 'Annualised Return': np.float64(-0.1871725616899125), 'Sharpe Ratio': np.float64(-0.2840349160841251), 'Max Drawdown': np.float64(-0.908383791282517)}


### Window=90

In [27]:
def cusum_mean_reversion90(df, window=90, h=0.05):
    df = df.copy()

    # rolling mean of returns
    df["mean_ret"] = df["Log_Return"].rolling(window).mean()

    # demeaned returns
    df["demeaned"] = df["Log_Return"] - df["mean_ret"]

    # rolling cumulative sum
    df["CUSUM"] = df["demeaned"].rolling(window).sum()

    position = []
    current_pos = 0

    for i in range(len(df)):

        c = df["CUSUM"].iloc[i]

        if np.isnan(c):
            position.append(0)
            continue

        # entry
        if current_pos == 0:
            if c > h:
                current_pos = -1
            elif c < -h:
                current_pos = 1

        # exit when back near zero
        elif current_pos == 1:
            if c >= 0:
                current_pos = 0

        elif current_pos == -1:
            if c <= 0:
                current_pos = 0

        position.append(current_pos)

    df["position"] = position

    df["strategy_return"] = df["position"].shift(1) * df["Log_Return"]

    return df

In [28]:
btc_cusum90 = cusum_mean_reversion90(btc, window=90, h=0.05)
eth_cusum90 = cusum_mean_reversion90(eth, window=90, h=0.05)

In [29]:
btc_results90 = evaluate_strategy(btc_cusum90["strategy_return"])
eth_results90 = evaluate_strategy(eth_cusum90["strategy_return"])

print("BTC 90 Results")
print(btc_results90)

print("\nETH 90 Results")
print(eth_results90)

BTC 90 Results
{'Cumulative PnL': np.float64(-0.9713326843086808), 'Average Daily PnL': np.float64(-0.001216437544951428), 'Volatility': np.float64(0.030852169837520798), 'Annualised Return': np.float64(-0.4439997039072712), 'Sharpe Ratio': np.float64(-0.8041664026625903), 'Max Drawdown': np.float64(-0.977270622205138)}

ETH 90 Results
{'Cumulative PnL': np.float64(-0.8641386600242553), 'Average Daily PnL': np.float64(-0.0006836029014903389), 'Volatility': np.float64(0.04147616462506317), 'Annualised Return': np.float64(-0.2495150590439737), 'Sharpe Ratio': np.float64(-0.35274444961437673), 'Max Drawdown': np.float64(-0.9614915534750585)}
